# Figure S15 and Table S3 — in-silico comparison of the instrument's BO with model-free sampling

Original benchmark `paperAnalysis/bo_sim` (vendored in `src/toofast_analysis/bo_sim/`, paths only), driving the platform code snapshot `third_party/toofast` in simulation. Oracle = nb23 spectral model trained on 274 runs rebuilt from raw/ (`toofast_analysis.bo_sim_data`), 27 landscapes (9 targets × posterior mean + 2 posterior samples) per cost function; arms: the BO as run (S28-M01 → `robust`, raw-spectrum cosine; S24-M01 → `prior_warm`, normalised PCA cosine), the instrument's projected LHS and uniform random sampling, extended to 400 experiments.

`toofast_analysis.figS15.main()` resumes from the caches in `data/processed/bo_sim/` (≈ 6 h from scratch: 540 BO runs). Outputs → `figures/bo_sim/` (`bench_extended` = S15, `bench_table_S3.csv` = Table S3).

## Analysis code
The code below is the code of the package (`src/toofast_analysis/`), copied in this notebook so that it can be read and run here.

### `src/toofast_analysis/paths.py`
*Copied from the package (do not edit here).*

In [ ]:
"""Repository paths."""
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())   # notebook: no __file__
# raw/ can live elsewhere (e.g. an extracted Zenodo archive): set TOOFAST_RAW
RAW = Path(os.environ.get("TOOFAST_RAW", ROOT / "raw"))
PROCESSED = ROOT / "data" / "processed"
INPUTS = ROOT / "data" / "inputs"      # frozen artefacts used by the platform during the runs (versioned, read-only)
RESOURCE_DIR = INPUTS
FIGURES = ROOT / "figures"


### `src/toofast_analysis/bo_sim_data.py`
*Copied from the package (do not edit here).*

In [ ]:
"""Training data of the in-silico benchmark oracle (S15, Table S3), rebuilt from raw/.

Original: 2FAST_ML/spectral_catalogue.py (nb20: flows, is_flat = std(last spectrum) < 0.03) and
2FAST_ML/spectral_selection.py (nb21: selection, tiers, last spectrum resampled on the 310-1000 nm grid of the first
selected file) -> spectral_selected.csv + spectra_matrix.npz. The selection itself (274 rows, with the recorded tiers)
is resources/bo_sim_oracle_files.csv; this module re-reads every H5 in raw/ and checks flows and is_flat against it.
"""
import csv

import h5py
import numpy as np
import pandas as pd


WL_RANGE = (310.0, 1000.0)
FLAT_STD_THR = 0.03
ABS_CANDS = ["spectra/extinction"]
WL_CANDS = ["spectra/wavelength"]


def selection():
    with open(RESOURCE_DIR / "bo_sim_oracle_files.csv", encoding="utf-8") as fh:
        return pd.read_csv(fh, float_precision="round_trip")


def read_spectrum(fp):
    """(wavelengths, LAST spectrum) — spectral_selection.read_spectrum."""
    with h5py.File(fp, "r") as f:
        loc = next((c for c in ABS_CANDS if c in f), None)
        wlk = next((w for w in WL_CANDS if w in f), None)
        if loc is None or wlk is None:
            return None, None
        return f[wlk][()], f[loc][-1]


def build(out_dir, raw=None):
    """Write spectral_selected.csv + spectra_matrix.npz (original names and columns) into out_dir."""
    from toofast_analysis.ml import catalog_2fast
    raw = RAW if raw is None else raw
    sel = selection()
    paths = [raw / p for p in sel.raw_path]
    wl0, _ = read_spectrum(paths[0])
    grid = wl0[(wl0 >= WL_RANGE[0]) & (wl0 <= WL_RANGE[1])]
    S, bad = [], []
    for p, (_, r) in zip(paths, sel.iterrows()):
        wl, a = read_spectrum(p)
        S.append(np.interp(grid, wl, a))
        rec = catalog_2fast.parse_h5_file(p)
        q = np.array([rec.Q1, rec.Q2, rec.Q3, rec.Q4, rec.Q5])
        if not np.allclose(q, r[["Q1", "Q2", "Q3", "Q4", "Q5"]].values.astype(float), atol=1e-9):
            bad.append((r.filename, "flows"))
        if bool(np.std(a) < FLAT_STD_THR) != bool(r.is_flat):
            bad.append((r.filename, "is_flat"))
    if bad:
        raise ValueError(f"raw/ disagrees with the recorded selection: {bad[:10]}")
    out_dir.mkdir(parents=True, exist_ok=True)
    sel.rename(columns={"raw_path": "filepath"}).assign(filepath=[str(p) for p in paths]).to_csv(
        out_dir / "spectral_selected.csv", index=False)
    np.savez(out_dir / "spectra_matrix.npz", S=np.vstack(S), wavelengths=grid)
    return out_dir


### `src/toofast_analysis/figS15.py`
*Copied from the package (do not edit here).*

In [ ]:
"""Figure S15 and Table S3 — in-silico comparison of the instrument's BO with model-free sampling.

Driver of the vendored benchmark (toofast_analysis.bo_sim = paperAnalysis/bo_sim, driving the vendored platform code
third_party/toofast in simulation). Steps, each cached under data/processed/bo_sim/:
  1. oracle training data rebuilt from raw/ (toofast_analysis.bo_sim_data; identical to 2FAST_ML/spectra_matrix.npz);
  2. oracle (nb23 spectral model + feasibility gate classifier+rule) and the 9-target panel (real target + 8 oracle
     targets, seed 0, sampled on the S28-M01 domain with no flow floor) — both identical to the originals;
  3. the BO runs actually used (S28-M01 -> 'robust', S24-M01 -> 'prior_warm'): 9 targets x 3 worlds x 10 seeds each
     (bo_sim.run_panel; ≈ 1 min per run alone, ≈ 4 min x 8 in parallel);
  4. paper_bench: regret curves, bootstrap statistics, model-free strategies extended to 400 experiments.
Outputs -> figures/bo_sim/ (bench_extended = S15; bench_metrics.csv + bench_table_S3.csv = Table S3).

    python -m toofast_analysis.figS15            (≈ 5–6 h from scratch; resumes from the caches)
"""
import warnings

import numpy as np
import pandas as pd

from toofast_analysis.bo_sim import baselines, bench, config as C, paper_bench as PB, targets as T
from toofast_analysis.bo_sim.oracle import get_oracle
from toofast_analysis.bo_sim.paper_figure import BO_OF_RUN

N_JOBS = 8


def prepare():
    if not C.SPECTRA_MATRIX.exists():
        build(C.SPECTRA_MATRIX.parent)
    orc = get_oracle()
    T.make_target_panel(orc, baselines.Sampler(baselines.run_bounds(C.RUN_S28, 0.0)))


def run_bo(run, n_jobs=N_JOBS):
    run_dir = C.BO_RUNS_DIR / run
    bench.run_panel(f"panel_{run.lower().split('_')[0]}", T.load_target_panel(), list(PB.WORLDS), [BO_OF_RUN[run]],
                    PB.SEEDS_BO, budget=PB.BUDGET, bounds=baselines.run_bounds(run_dir, 0.0), run_dir=run_dir,
                    prior_name=C.DEFAULT_PRIOR, n_restarts=None, gate="classifier+rule", n_jobs=n_jobs, noise=0.0)


def table_s3(cases, exts):
    """Median normalised regret at N = 30 over the paired 30-experiment runs (= bench_metrics `regret_median`, the source of
    Table S3: 10 seeds per landscape for bo / instr, 100 for unif) and smallest N with P(top 1 %) >= 50 % (bo: its 30
    runs; instr / unif: the runs extended to 400 experiments, 100 seeds — as in bench_extended = S15)."""
    rows = []
    for case, ext in zip(cases, exts):
        hits = {"bo": np.concatenate([c["bo"] <= c["thr1"] for c in case["cells"].values()]),
                **{k: np.concatenate([c[k] <= c["thr1"] for c in ext["cells"].values()]) for k in ("instr", "unif")}}
        series = {k: np.concatenate([c[k] for c in case["cells"].values()]) for k in PB.ARMS}
        for a in PB.ARMS:
            rows.append(dict(run=case["run"], arm=a, median_regret_30=float(np.median(series[a][:, PB.BUDGET - 1])),
                             n_for_p_top1_50=PB._n50(hits[a].mean(0))))
    return pd.DataFrame(rows)


def main():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    warnings.filterwarnings("ignore")
    prepare()
    for run in PB.RUNS:
        run_bo(run)
    C.FIG_OUT.mkdir(parents=True, exist_ok=True)
    cases = [PB.build_case(r) for r in PB.RUNS]
    boots = [PB.bootstrap(c) for c in cases]
    pd.concat([PB.metrics(c, b) for c, b in zip(cases, boots)]).to_csv(C.FIG_OUT / "bench_metrics.csv", index=False)
    pd.concat([PB.per_target_table(c) for c in cases]).to_csv(C.FIG_OUT / "bench_per_target.csv", index=False)
    PB.oracle_fidelity().to_csv(C.FIG_OUT / "bench_oracle_fidelity.csv", index=False)
    PB.fig_curves(cases, boots, "bench_curves"); plt.close("all")
    PB.fig_time_to_target(cases, "bench_time_to_target"); plt.close("all")
    exts = [PB.build_extended(r) for r in PB.RUNS]
    PB.fig_extended(cases, exts); plt.close("all")
    t = table_s3(cases, exts)
    t.to_csv(C.FIG_OUT / "bench_table_S3.csv", index=False)
    print(t.to_string(index=False))
    return t


## Figure

In [ ]:
from IPython.display import Image, display
from toofast_analysis.bo_sim import config as C

table = main()
table


In [ ]:
display(Image(str(C.FIG_OUT / 'bench_extended.png')))
